# Model Evaluation and Hybrid Weight Tuning

## Project
An Explainable Knowledge Graph-Driven Hybrid Recommender for Intelligent Task Allocation in Distributed Onboarding Operations

## Objective

The purpose of this notebook is to evaluate the recommendation model developed in the previous notebook.

The evaluation will:

- Test different hybrid weight combinations
- Use the validation dataset to select the best weights
- Compare the hybrid recommender with simpler recommendation approaches
- Evaluate the final model using unseen testing data
- Measure Precision@5, Recall@5 and MAP@5
- Examine workload distribution
- Assess explanation coverage

The validation dataset will be used for model tuning, while the testing dataset will only be used for the final evaluation.

## 1. Mounting Google Drive

Google Drive is mounted so that the processed datasets and recommendation model components created in the previous notebooks can be accessed.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 2. Importing Required Libraries

The required Python libraries are imported for data handling, numerical calculations, model loading and recommendation evaluation.

In [ ]:
import os
import json
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

## 3. Defining Project Paths

The project paths are defined to access the processed datasets, training splits and saved recommender components.

In [ ]:
BASE_PATH = (
    "/content/drive/MyDrive/"
    "task-allocation-recommender"
)

PROCESSED_PATH = os.path.join(
    BASE_PATH,
    "data",
    "onboarding",
    "processed"
)

SPLITS_PATH = os.path.join(
    BASE_PATH,
    "data",
    "onboarding",
    "splits"
)

MODEL_PATH = os.path.join(
    BASE_PATH,
    "models"
)

print("Processed:", PROCESSED_PATH)
print("Splits:", SPLITS_PATH)
print("Models:", MODEL_PATH)

Processed: /content/drive/MyDrive/task-allocation-recommender/data/onboarding/processed
Splits: /content/drive/MyDrive/task-allocation-recommender/data/onboarding/splits
Models: /content/drive/MyDrive/task-allocation-recommender/models


## 4. Loading Evaluation Data

The validation and testing datasets created during preprocessing are loaded.

The validation dataset contains 15% of the historical assignments and will be used to tune the hybrid recommendation weights.

The testing dataset contains the final 15% and will remain unseen until the final model has been selected.

In [ ]:
tasks = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "onboarding_tasks.parquet"
    )
)

validation_data = pd.read_parquet(
    os.path.join(
        SPLITS_PATH,
        "validation.parquet"
    )
)

test_data = pd.read_parquet(
    os.path.join(
        SPLITS_PATH,
        "test.parquet"
    )
)

In [ ]:
print("Tasks:", tasks.shape)
print("Validation:", validation_data.shape)
print("Testing:", test_data.shape)

Tasks: (100000, 16)
Validation: (15000, 9)
Testing: (15000, 9)


## 5. Loading the Trained Recommender Components

The recommendation components developed in Notebook 02 are loaded from the saved Joblib file.

This allows the evaluation notebook to reuse the trained TF-IDF model, collaborative filtering information, knowledge graph information and initial hybrid settings without rebuilding them.

In [ ]:
model_file = os.path.join(
    MODEL_PATH,
    "recommender_components.joblib"
)

recommender_components = joblib.load(
    model_file
)

print(
    "Loaded:",
    model_file
)

Loaded: /content/drive/MyDrive/task-allocation-recommender/models/recommender_components.joblib


In [ ]:
print(
    recommender_components.keys()
)

dict_keys(['tfidf_vectorizer', 'agent_skill_vectors', 'agents', 'interaction_matrix', 'agent_similarity', 'collaborative_scores', 'profile_to_column', 'historical_task_types', 'knowledge_graph', 'initial_weights'])


In [ ]:
tfidf_vectorizer = (
    recommender_components[
        "tfidf_vectorizer"
    ]
)

agent_skill_vectors = (
    recommender_components[
        "agent_skill_vectors"
    ]
)

agents = (
    recommender_components[
        "agents"
    ].copy()
)

collaborative_scores = (
    recommender_components[
        "collaborative_scores"
    ]
)

profile_to_column = (
    recommender_components[
        "profile_to_column"
    ]
)

historical_task_types = (
    recommender_components[
        "historical_task_types"
    ]
)

G = (
    recommender_components[
        "knowledge_graph"
    ]
)

INITIAL_WEIGHTS = (
    recommender_components[
        "initial_weights"
    ]
)

In [ ]:
print(
    "Agents:",
    len(agents)
)

print(
    "Knowledge graph nodes:",
    G.number_of_nodes()
)

print(
    "Knowledge graph edges:",
    G.number_of_edges()
)

print(
    "Initial weights:",
    INITIAL_WEIGHTS
)

Agents: 500
Knowledge graph nodes: 520
Knowledge graph edges: 2893
Initial weights: {'content': 0.35, 'collaborative': 0.25, 'knowledge_graph': 0.25, 'operational': 0.15}


In [ ]:
tasks["task_id"] = (
    tasks["task_id"]
    .astype(str)
)

agents["agent_id"] = (
    agents["agent_id"]
    .astype(str)
)

validation_data["task_id"] = (
    validation_data["task_id"]
    .astype(str)
)

validation_data["agent_id"] = (
    validation_data["agent_id"]
    .astype(str)
)

test_data["task_id"] = (
    test_data["task_id"]
    .astype(str)
)

test_data["agent_id"] = (
    test_data["agent_id"]
    .astype(str)
)

In [ ]:
print(
    "Validation records:",
    len(validation_data)
)

print(
    "Testing records:",
    len(test_data)
)

print(
    "Agents:",
    len(agents)
)

Validation records: 15000
Testing records: 15000
Agents: 500


### Evaluation Data Result

The validation and testing datasets were successfully loaded.

The saved recommendation components from Notebook 02 were also restored successfully.

The system is now ready for hybrid weight tuning and model evaluation.

The validation dataset will be used first. The testing dataset will remain untouched until the best hybrid configuration has been selected.

## 6. Preparing Evaluation Features

Before evaluating the recommender, the operational and knowledge graph features required for scoring are reconstructed.

These include:

- Agent availability
- Current workload
- Historical performance
- Historical experience with each task type

The same scoring logic used during model development is maintained so that validation results are comparable.

In [ ]:
evaluation_agents = agents.copy()

evaluation_agents[
    "availability_score"
] = (
    evaluation_agents[
        "availability_status"
    ]
    .astype(str)
    .str.lower()
    .eq("available")
    .astype(float)
)

In [ ]:
max_workload = (
    evaluation_agents[
        "current_workload"
    ].max()
)

if max_workload == 0:
    max_workload = 1

evaluation_agents[
    "workload_score"
] = (
    1
    -
    (
        evaluation_agents[
            "current_workload"
        ]
        / max_workload
    )
)

In [ ]:
evaluation_agents[
    "performance_score"
] = (
    evaluation_agents[
        "performance_score"
    ]
    .fillna(
        evaluation_agents[
            "performance_score"
        ].mean()
    )
    .clip(0, 1)
)

In [ ]:
evaluation_agents[
    "operational_score"
] = (
    0.40
    * evaluation_agents[
        "availability_score"
    ]

    + 0.35
    * evaluation_agents[
        "workload_score"
    ]

    + 0.25
    * evaluation_agents[
        "performance_score"
    ]
)

In [ ]:
evaluation_agents[
    [
        "agent_id",
        "availability_score",
        "workload_score",
        "performance_score",
        "operational_score"
    ]
].head()

,agent_id,availability_score,workload_score,performance_score,operational_score
0,AGT-0001,1.0,1.0,0.489,0.87225
1,AGT-0002,0.0,0.0,0.512,0.12800
2,AGT-0003,1.0,0.4,0.512,0.66800
3,AGT-0004,1.0,0.4,0.498,0.66450
4,AGT-0005,1.0,0.6,0.576,0.75400


## 7. Preparing Task Information

The task dataset is indexed using the task ID so that each validation or testing assignment can quickly retrieve the corresponding task characteristics.

In [ ]:
task_lookup = (
    tasks
    .set_index(
        "task_id",
        drop=False
    )
)

In [ ]:
print(
    "Tasks available for evaluation:",
    len(task_lookup)
)

Tasks available for evaluation: 100000


### Skill Matching Function

A helper function is used to calculate the proportion of task-required skills possessed by an agent.

A score of 1 means that the agent has all the skills required by the task.

In [ ]:
def split_skills(skill_text):

    if pd.isna(skill_text):
        return []

    return [
        skill.strip()
        for skill
        in str(skill_text).split(",")
        if skill.strip()
    ]

In [ ]:
def calculate_skill_match(
    task_skills,
    agent_skills
):

    required = set(
        split_skills(
            task_skills
        )
    )

    available = set(
        split_skills(
            agent_skills
        )
    )

    if len(required) == 0:
        return 0

    matched = (
        required.intersection(
            available
        )
    )

    return (
        len(matched)
        / len(required)
    )

## 8. Recommendation Scoring Function

A scoring function is created to generate recommendation scores for all agents for a given onboarding task.

The function calculates:

- Content-based score
- Collaborative filtering score
- Knowledge graph score
- Operational score
- Final hybrid score

This function will be used repeatedly during validation and testing.

In [ ]:
def score_task(
    task_row,
    weights
):

    # --------------------------------
    # 1. Content-Based Score
    # --------------------------------

    task_vector = (
        tfidf_vectorizer
        .transform(
            [
                str(
                    task_row[
                        "required_skills"
                    ]
                )
            ]
        )
    )

    content_score = (
        cosine_similarity(
            task_vector,
            agent_skill_vectors
        )
        .flatten()
    )

    # --------------------------------
    # 2. Collaborative Score
    # --------------------------------

    task_profile = (
        task_row[
            "task_profile"
        ]
    )

    if (
        task_profile
        in profile_to_column
    ):

        profile_column = (
            profile_to_column[
                task_profile
            ]
        )

        cf_score = (
            collaborative_scores[
                :,
                profile_column
            ]
        )

    else:

        cf_score = np.zeros(
            len(evaluation_agents)
        )

    # --------------------------------
    # 3. Knowledge Graph Score
    # --------------------------------

    skill_scores = np.array(
        [
            calculate_skill_match(
                task_row[
                    "required_skills"
                ],
                agent_skills
            )

            for agent_skills
            in evaluation_agents[
                "skills"
            ]
        ]
    )

    location_scores = (
        evaluation_agents[
            "county"
        ]
        .astype(str)
        .eq(
            str(
                task_row[
                    "county"
                ]
            )
        )
        .astype(float)
        .to_numpy()
    )

    history = (
        historical_task_types[
            historical_task_types[
                "task_type"
            ]
            ==
            task_row[
                "task_type"
            ]
        ]
        .set_index(
            "agent_id"
        )["count"]
    )

    historical_counts = (
        evaluation_agents[
            "agent_id"
        ]
        .map(history)
        .fillna(0)
        .to_numpy()
    )

    max_history = (
        historical_counts.max()
    )

    if max_history == 0:
        max_history = 1

    task_type_scores = (
        historical_counts
        / max_history
    )

    kg_score = (
        0.60
        * skill_scores

        + 0.20
        * location_scores

        + 0.20
        * task_type_scores
    )

    # --------------------------------
    # 4. Operational Score
    # --------------------------------

    operational_score = (
        evaluation_agents[
            "operational_score"
        ]
        .to_numpy()
    )

    # --------------------------------
    # 5. Hybrid Score
    # --------------------------------

    hybrid_score = (
        weights["content"]
        * content_score

        + weights[
            "collaborative"
        ]
        * cf_score

        + weights[
            "knowledge_graph"
        ]
        * kg_score

        + weights[
            "operational"
        ]
        * operational_score
    )

    return {
        "content":
            content_score,

        "collaborative":
            cf_score,

        "knowledge_graph":
            kg_score,

        "operational":
            operational_score,

        "hybrid":
            hybrid_score
    }

In [ ]:
validation_example = (
    validation_data.iloc[0]
)

validation_task_id = (
    validation_example[
        "task_id"
    ]
)

validation_task = (
    task_lookup.loc[
        validation_task_id
    ]
)

In [ ]:
test_scores = score_task(
    validation_task,
    INITIAL_WEIGHTS
)

In [ ]:
print(
    "Content scores:",
    len(
        test_scores[
            "content"
        ]
    )
)

print(
    "Collaborative scores:",
    len(
        test_scores[
            "collaborative"
        ]
    )
)

print(
    "Knowledge Graph scores:",
    len(
        test_scores[
            "knowledge_graph"
        ]
    )
)

print(
    "Hybrid scores:",
    len(
        test_scores[
            "hybrid"
        ]
    )
)

Content scores: 500
Collaborative scores: 500
Knowledge Graph scores: 500
Hybrid scores: 500


In [ ]:
top_indices = np.argsort(
    -test_scores[
        "hybrid"
    ]
)[:5]

In [ ]:
validation_top5 = (
    evaluation_agents
    .iloc[
        top_indices
    ][
        [
            "agent_id",
            "county",
            "skills",
            "availability_status",
            "current_workload"
        ]
    ]
    .copy()
)

In [ ]:
validation_top5[
    "hybrid_score"
] = (
    test_scores[
        "hybrid"
    ][
        top_indices
    ]
)

In [ ]:
validation_top5.insert(
    0,
    "rank",
    range(1, 6)
)

validation_top5

,rank,agent_id,county,skills,availability_status,current_workload,hybrid_score
385,1,AGT-0386,Uasin Gishu,"Customer Onboarding, KYC Verification",Available,0,0.979173
376,2,AGT-0377,Uasin Gishu,"Customer Onboarding, KYC Verification",Available,2,0.956898
165,3,AGT-0166,Kisumu,"Customer Onboarding, KYC Verification",Available,0,0.930593
72,4,AGT-0073,Nyeri,"Customer Onboarding, KYC Verification",Available,0,0.926301
464,5,AGT-0465,Mombasa,"Customer Onboarding, KYC Verification",Available,0,0.922113


In [ ]:
print(
    "Actual historical agent:",
    validation_example[
        "agent_id"
    ]
)

print(
    "\nRecommended Top 5:"
)

print(
    validation_top5[
        "agent_id"
    ].tolist()
)

Actual historical agent: AGT-0342

Recommended Top 5:
['AGT-0386', 'AGT-0377', 'AGT-0166', 'AGT-0073', 'AGT-0465']


## 9. Evaluation Approach

For each task in the validation and testing datasets, the recommender produces a ranked list of agents.

The agent recorded in the historical assignment is treated as the relevant agent for that task.

The evaluation then checks whether the historical agent appears among the Top-K recommended agents.

The main ranking metrics used are:

- Precision@K
- Recall@K
- MAP@K

Because each task in the historical dataset has one recorded assigned agent, Recall@K indicates whether that agent appears within the Top-K recommendations.

MAP@K additionally rewards the system when the relevant agent appears closer to the top of the ranked list.

## 10. Ranking Metrics

The recommendation model is evaluated using ranking metrics.

### Precision@5
Measures how much of the Top-5 recommendation list is relevant.

### Recall@5
Measures whether the historically assigned agent appears within the Top-5 recommendations.

### MAP@5
Mean Average Precision considers both whether the correct agent appears in the Top-5 and how highly the agent is ranked.

A higher score indicates better recommendation performance.

In [ ]:
def calculate_ranking_result(
    recommended_agents,
    actual_agent,
    k=5
):

    top_k = recommended_agents[:k]

    if actual_agent in top_k:

        rank = (
            top_k.index(actual_agent)
            + 1
        )

        hit = 1

        precision = 1 / k
        recall = 1
        average_precision = 1 / rank

    else:

        rank = None

        hit = 0
        precision = 0
        recall = 0
        average_precision = 0

    return {
        "hit": hit,
        "precision": precision,
        "recall": recall,
        "average_precision": average_precision,
        "rank": rank
    }

## 11. Recommendation Score Caching

Many onboarding tasks share the same task characteristics.

To improve evaluation speed, recommendation scores are cached using the task profile, task type, county and required skills.

When another task has the same characteristics, the previously calculated recommendation scores can be reused.

In [ ]:
score_cache = {}

In [ ]:
def get_cached_scores(
    task_row,
    weights
):

    cache_key = (
        str(task_row["task_profile"]),
        str(task_row["task_type"]),
        str(task_row["county"]),
        str(task_row["required_skills"]),
        tuple(
            sorted(
                weights.items()
            )
        )
    )

    if cache_key not in score_cache:

        score_cache[
            cache_key
        ] = score_task(
            task_row,
            weights
        )

    return score_cache[
        cache_key
    ]

## 12. Evaluating the Validation Dataset

Each validation task is passed through the hybrid recommender.

For every task, the Top-5 recommended agents are compared with the historical assigned agent.

The results are then averaged to calculate Precision@5, Recall@5 and MAP@5.

In [ ]:
def evaluate_model(
    evaluation_data,
    weights,
    k=5
):

    results = []

    for _, record in evaluation_data.iterrows():

        task_id = str(
            record["task_id"]
        )

        actual_agent = str(
            record["agent_id"]
        )

        task_row = (
            task_lookup.loc[
                task_id
            ]
        )

        scores = get_cached_scores(
            task_row,
            weights
        )

        top_indices = np.argsort(
            -scores["hybrid"]
        )[:k]

        recommended_agents = (
            evaluation_agents
            .iloc[top_indices][
                "agent_id"
            ]
            .astype(str)
            .tolist()
        )

        result = (
            calculate_ranking_result(
                recommended_agents,
                actual_agent,
                k
            )
        )

        results.append(
            result
        )

    results_df = pd.DataFrame(
        results
    )

    metrics = {
        f"Precision@{k}":
            results_df[
                "precision"
            ].mean(),

        f"Recall@{k}":
            results_df[
                "recall"
            ].mean(),

        f"MAP@{k}":
            results_df[
                "average_precision"
            ].mean()
    }

    return metrics, results_df

In [ ]:
validation_small = (
    validation_data
    .head(500)
    .copy()
)

In [ ]:
start_time = time.time()

small_metrics, small_results = (
    evaluate_model(
        validation_small,
        INITIAL_WEIGHTS,
        k=5
    )
)

elapsed_time = (
    time.time()
    - start_time
)

In [ ]:
print(
    "INITIAL VALIDATION TEST"
)

for metric, value in small_metrics.items():

    print(
        metric,
        ":",
        round(value, 4)
    )

print(
    "\nEvaluation time:",
    round(
        elapsed_time,
        2
    ),
    "seconds"
)

INITIAL VALIDATION TEST
Precision@5 : 0.054
Recall@5 : 0.27
MAP@5 : 0.1721

Evaluation time: 1.91 seconds


### Evaluation Limitation

The historical task-agent assignments used in this experiment are synthetic.

For each task, only one agent is recorded as the historical assigned agent even though several agents may have been equally suitable.

Therefore, the ranking metrics measure how well the recommender reproduces the simulated assignment pattern rather than proving real-world organisational performance.

The results are interpreted as prototype experimental results.

## 13. Hybrid Weight Tuning

The initial hybrid recommender used manually selected starting weights.

To avoid assuming that these initial weights are optimal, different weight combinations are evaluated using the validation dataset.

The validation dataset is used only for model selection and tuning.

The testing dataset remains untouched until the best weight combination has been selected.

The weight combination with the strongest Recall@5 and MAP@5 performance will be selected for the final hybrid recommender.

In [ ]:
validation_tuning = (
    validation_data
    .sample(
        n=3000,
        random_state=42
    )
    .reset_index(drop=True)
)

print(
    "Validation records for tuning:",
    len(validation_tuning)
)

Validation records for tuning: 3000


In [ ]:
weight_candidates = [
    {
        "name": "Initial",
        "content": 0.35,
        "collaborative": 0.25,
        "knowledge_graph": 0.25,
        "operational": 0.15
    },

    {
        "name": "Content Focus",
        "content": 0.45,
        "collaborative": 0.20,
        "knowledge_graph": 0.20,
        "operational": 0.15
    },

    {
        "name": "Collaborative Focus",
        "content": 0.25,
        "collaborative": 0.40,
        "knowledge_graph": 0.20,
        "operational": 0.15
    },

    {
        "name": "Knowledge Graph Focus",
        "content": 0.25,
        "collaborative": 0.20,
        "knowledge_graph": 0.40,
        "operational": 0.15
    },

    {
        "name": "Operational Focus",
        "content": 0.25,
        "collaborative": 0.20,
        "knowledge_graph": 0.25,
        "operational": 0.30
    },

    {
        "name": "Balanced",
        "content": 0.25,
        "collaborative": 0.25,
        "knowledge_graph": 0.25,
        "operational": 0.25
    },

    {
        "name": "Content + KG",
        "content": 0.35,
        "collaborative": 0.15,
        "knowledge_graph": 0.35,
        "operational": 0.15
    },

    {
        "name": "CF + KG",
        "content": 0.20,
        "collaborative": 0.30,
        "knowledge_graph": 0.35,
        "operational": 0.15
    }
]

In [ ]:
for candidate in weight_candidates:

    total = (
        candidate["content"]
        + candidate["collaborative"]
        + candidate["knowledge_graph"]
        + candidate["operational"]
    )

    print(
        candidate["name"],
        "=",
        round(total, 2)
    )

Initial = 1.0
Content Focus = 1.0
Collaborative Focus = 1.0
Knowledge Graph Focus = 1.0
Operational Focus = 1.0
Balanced = 1.0
Content + KG = 1.0
CF + KG = 1.0


### Testing Candidate Weight Combinations

Each candidate weight combination is evaluated on the same validation sample.

Using the same records allows the different hybrid configurations to be compared fairly.

In [ ]:
tuning_results = []

for candidate in weight_candidates:

    weights = {
        "content":
            candidate["content"],

        "collaborative":
            candidate["collaborative"],

        "knowledge_graph":
            candidate["knowledge_graph"],

        "operational":
            candidate["operational"]
    }

    start_time = time.time()

    metrics, _ = evaluate_model(
        validation_tuning,
        weights,
        k=5
    )

    elapsed = (
        time.time()
        - start_time
    )

    tuning_results.append(
        {
            "configuration":
                candidate["name"],

            "content":
                candidate["content"],

            "collaborative":
                candidate["collaborative"],

            "knowledge_graph":
                candidate[
                    "knowledge_graph"
                ],

            "operational":
                candidate["operational"],

            "Precision@5":
                metrics["Precision@5"],

            "Recall@5":
                metrics["Recall@5"],

            "MAP@5":
                metrics["MAP@5"],

            "time_seconds":
                elapsed
        }
    )

    print(
        candidate["name"],
        "completed"
    )

Initial completed
Content Focus completed
Collaborative Focus completed
Knowledge Graph Focus completed
Operational Focus completed
Balanced completed
Content + KG completed
CF + KG completed


In [ ]:
tuning_results_df = (
    pd.DataFrame(
        tuning_results
    )
)

In [ ]:
tuning_results_df = (
    tuning_results_df
    .sort_values(
        [
            "Recall@5",
            "MAP@5"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

tuning_results_df

,configuration,content,collaborative,knowledge_graph,operational,Precision@5,Recall@5,MAP@5,time_seconds
0,Knowledge Graph Focus,0.25,0.20,0.40,0.15,0.069667,0.348333,0.218444,3.875026
1,CF + KG,0.20,0.30,0.35,0.15,0.064267,0.321333,0.187817,4.563891
2,Content + KG,0.35,0.15,0.35,0.15,0.060667,0.303333,0.199194,3.968630
3,Initial,0.35,0.25,0.25,0.15,0.052867,0.264333,0.165239,2.698472
4,Balanced,0.25,0.25,0.25,0.25,0.051333,0.256667,0.153294,5.384070
5,Content Focus,0.45,0.20,0.20,0.15,0.050333,0.251667,0.156917,4.554837
6,Operational Focus,0.25,0.20,0.25,0.30,0.048133,0.240667,0.150711,3.891024
7,Collaborative Focus,0.25,0.40,0.20,0.15,0.045733,0.228667,0.132722,4.904051


In [ ]:
best_configuration = (
    tuning_results_df.iloc[0]
)

BEST_WEIGHTS = {
    "content":
        float(
            best_configuration[
                "content"
            ]
        ),

    "collaborative":
        float(
            best_configuration[
                "collaborative"
            ]
        ),

    "knowledge_graph":
        float(
            best_configuration[
                "knowledge_graph"
            ]
        ),

    "operational":
        float(
            best_configuration[
                "operational"
            ]
        )
}

In [ ]:
print(
    "BEST CONFIGURATION:",
    best_configuration[
        "configuration"
    ]
)

print(
    "\nBest weights:",
    BEST_WEIGHTS
)

print(
    "\nRecall@5:",
    round(
        best_configuration[
            "Recall@5"
        ],
        4
    )
)

print(
    "MAP@5:",
    round(
        best_configuration[
            "MAP@5"
        ],
        4
    )
)

BEST CONFIGURATION: Knowledge Graph Focus

Best weights: {'content': 0.25, 'collaborative': 0.2, 'knowledge_graph': 0.4, 'operational': 0.15}

Recall@5: 0.3483
MAP@5: 0.2184


### Hybrid Weight Tuning Result

Several hybrid weight combinations were evaluated using the validation dataset.

The configurations changed the relative importance of content-based filtering, collaborative filtering, knowledge graph reasoning and operational factors.

The best-performing configuration was selected based mainly on Recall@5 and MAP@5.

This selected configuration will be used for the final evaluation on the unseen testing dataset.

# Final Model Evaluation

## 14. Evaluation on the Unseen Testing Dataset

After selecting the best hybrid weights using the validation dataset, the final model is evaluated using the testing dataset.

The testing dataset contains 15,000 historical task-agent assignments that were not used during model development or weight tuning.

This provides a final estimate of how well the recommender performs on unseen data.

The selected hybrid weights are:

- Content-Based Filtering: 25%
- Collaborative Filtering: 20%
- Knowledge Graph: 40%
- Operational Factors: 15%

In [ ]:
score_cache = {}

In [ ]:
start_time = time.time()

final_test_metrics, final_test_results = (
    evaluate_model(
        test_data,
        BEST_WEIGHTS,
        k=5
    )
)

test_time = (
    time.time()
    - start_time
)

In [ ]:
print("FINAL TEST RESULTS")

for metric, value in final_test_metrics.items():

    print(
        metric,
        ":",
        round(value, 4)
    )

print(
    "\nTesting records:",
    len(test_data)
)

print(
    "Total evaluation time:",
    round(test_time, 2),
    "seconds"
)

FINAL TEST RESULTS
Precision@5 : 0.07
Recall@5 : 0.3499
MAP@5 : 0.2186

Testing records: 15000
Total evaluation time: 12.63 seconds


In [ ]:
successful_top5 = (
    final_test_results[
        "hit"
    ].sum()
)

print(
    "Tasks where historical agent "
    "appeared in Top 5:",
    successful_top5
)

print(
    "Total test tasks:",
    len(final_test_results)
)

Tasks where historical agent appeared in Top 5: 5248
Total test tasks: 15000


### Final Testing Result

The final hybrid recommendation model was evaluated on the unseen testing dataset after model tuning was completed.

Precision@5, Recall@5 and MAP@5 were used to evaluate the quality of the ranked agent recommendations.

Recall@5 shows how often the historically assigned agent appeared within the five agents recommended by the system.

MAP@5 additionally considers the position of the historical agent within the recommendation list, giving a higher score when the relevant agent appears closer to rank one.

The testing results are treated as prototype experimental results because the historical task-agent assignments were synthetically generated.

## 15. Comparison with Individual Recommendation Components

The final hybrid recommender is compared with individual recommendation components.

The comparison includes:

- Content-Based Filtering only
- Collaborative Filtering only
- Knowledge Graph only
- Final Hybrid Recommender

All models are evaluated using the same unseen testing dataset to ensure a fair comparison.

In [ ]:
comparison_models = {
    "Content-Based": {
        "content": 1.0,
        "collaborative": 0.0,
        "knowledge_graph": 0.0,
        "operational": 0.0
    },

    "Collaborative": {
        "content": 0.0,
        "collaborative": 1.0,
        "knowledge_graph": 0.0,
        "operational": 0.0
    },

    "Knowledge Graph": {
        "content": 0.0,
        "collaborative": 0.0,
        "knowledge_graph": 1.0,
        "operational": 0.0
    },

    "Hybrid": BEST_WEIGHTS
}

In [ ]:
model_comparison = []

for model_name, weights in comparison_models.items():

    score_cache = {}

    start = time.time()

    metrics, _ = evaluate_model(
        test_data,
        weights,
        k=5
    )

    elapsed = (
        time.time()
        - start
    )

    model_comparison.append(
        {
            "Model": model_name,

            "Precision@5":
                metrics["Precision@5"],

            "Recall@5":
                metrics["Recall@5"],

            "MAP@5":
                metrics["MAP@5"],

            "Evaluation_Time":
                elapsed
        }
    )

    print(
        model_name,
        "completed"
    )

Content-Based completed
Collaborative completed
Knowledge Graph completed
Hybrid completed


In [ ]:
comparison_df = pd.DataFrame(
    model_comparison
)

comparison_df = (
    comparison_df
    .sort_values(
        "Recall@5",
        ascending=False
    )
    .reset_index(drop=True)
)

comparison_df

,Model,Precision@5,Recall@5,MAP@5,Evaluation_Time
0,Knowledge Graph,0.093133,0.465667,0.218577,12.107295
1,Hybrid,0.069973,0.349867,0.218569,12.913986
2,Content-Based,0.016840,0.084200,0.037798,14.221859
3,Collaborative,0.015587,0.077933,0.036268,12.091757


### Model Comparison Interpretation

The Knowledge Graph model achieved the highest Recall@5 on the unseen testing dataset, followed by the Hybrid model.

The Knowledge Graph achieved a Recall@5 of approximately 0.466, meaning that the historically assigned agent appeared within the Top-5 recommendations for about 46.6% of the testing tasks.

The Hybrid model achieved a Recall@5 of approximately 0.350.

The Content-Based and Collaborative Filtering models produced considerably lower ranking performance when used independently.

The results suggest that relationships between agent skills, task requirements, location and historical task-type experience were particularly important in the simulated onboarding task allocation environment.

Although the Hybrid model did not outperform the Knowledge Graph component on Recall@5, it combines multiple recommendation signals and operational factors such as availability, workload and performance. This allows the system to consider both recommendation relevance and practical task allocation requirements.

These findings are based on synthetically generated historical assignments and therefore represent prototype experimental results rather than real organisational performance.

In [ ]:
hybrid_successful_tasks = int(
    round(
        final_test_metrics["Recall@5"]
        * len(test_data)
    )
)

print(
    "Hybrid Top-5 successful tasks:",
    hybrid_successful_tasks
)

print(
    "Total testing tasks:",
    len(test_data)
)

print(
    "Hybrid Recall@5:",
    round(
        final_test_metrics["Recall@5"] * 100,
        2
    ),
    "%"
)

Hybrid Top-5 successful tasks: 5248
Total testing tasks: 15000
Hybrid Recall@5: 34.99 %


## 16. Workload Distribution Evaluation

Task allocation should not only recommend suitable agents but should also avoid repeatedly recommending the same small group of agents.

To examine workload distribution, the Top-1 recommended agent is recorded for each testing task.

The standard deviation of recommendation counts across agents is then calculated.

A lower standard deviation indicates that recommendations are distributed more evenly among the available agents.

In [ ]:
def calculate_recommendation_distribution(
    evaluation_data,
    weights
):

    recommendation_counts = {
        agent_id: 0
        for agent_id
        in evaluation_agents[
            "agent_id"
        ]
    }

    local_cache = {}

    for _, record in evaluation_data.iterrows():

        task_id = str(
            record["task_id"]
        )

        task_row = task_lookup.loc[
            task_id
        ]

        cache_key = (
            str(task_row["task_profile"]),
            str(task_row["task_type"]),
            str(task_row["county"]),
            str(task_row["required_skills"])
        )

        if cache_key not in local_cache:

            local_cache[cache_key] = score_task(
                task_row,
                weights
            )

        scores = local_cache[
            cache_key
        ]["hybrid"]

        best_index = int(
            np.argmax(scores)
        )

        recommended_agent = (
            evaluation_agents.iloc[
                best_index
            ]["agent_id"]
        )

        recommendation_counts[
            recommended_agent
        ] += 1

    return recommendation_counts

In [ ]:
kg_only_weights = {
    "content": 0.0,
    "collaborative": 0.0,
    "knowledge_graph": 1.0,
    "operational": 0.0
}

In [ ]:
kg_distribution = (
    calculate_recommendation_distribution(
        test_data,
        kg_only_weights
    )
)

hybrid_distribution = (
    calculate_recommendation_distribution(
        test_data,
        BEST_WEIGHTS
    )
)

In [ ]:
kg_counts = np.array(
    list(
        kg_distribution.values()
    )
)

hybrid_counts = np.array(
    list(
        hybrid_distribution.values()
    )
)

kg_workload_std = (
    kg_counts.std()
)

hybrid_workload_std = (
    hybrid_counts.std()
)

print(
    "Knowledge Graph recommendation "
    "distribution std:",
    round(
        kg_workload_std,
        2
    )
)

print(
    "Hybrid recommendation "
    "distribution std:",
    round(
        hybrid_workload_std,
        2
    )
)

Knowledge Graph recommendation distribution std: 106.06
Hybrid recommendation distribution std: 80.72


## 17. Explanation Coverage

Explanation coverage measures whether the recommended agent can be supported by at least one understandable relationship or operational reason.

Possible explanation factors include:

- Matching required skills
- Same location as the task
- Historical experience with the same task type
- Current availability
- Low workload
- Historical performance

In [ ]:
def has_explanation(
    task_row,
    agent_row
):

    task_skills = set(
        split_skills(
            task_row[
                "required_skills"
            ]
        )
    )

    agent_skills = set(
        split_skills(
            agent_row[
                "skills"
            ]
        )
    )

    skill_match = len(
        task_skills.intersection(
            agent_skills
        )
    ) > 0

    location_match = (
        str(
            task_row["county"]
        )
        ==
        str(
            agent_row["county"]
        )
    )

    available = (
        str(
            agent_row[
                "availability_status"
            ]
        ).lower()
        == "available"
    )

    return (
        skill_match
        or location_match
        or available
    )

In [ ]:
explanation_count = 0

explanation_cache = {}

for _, record in test_data.iterrows():

    task_row = task_lookup.loc[
        str(record["task_id"])
    ]

    cache_key = (
        str(task_row["task_profile"]),
        str(task_row["task_type"]),
        str(task_row["county"]),
        str(task_row["required_skills"])
    )

    if cache_key not in explanation_cache:

        explanation_cache[
            cache_key
        ] = score_task(
            task_row,
            BEST_WEIGHTS
        )

    scores = explanation_cache[
        cache_key
    ]["hybrid"]

    best_index = int(
        np.argmax(scores)
    )

    best_agent = (
        evaluation_agents.iloc[
            best_index
        ]
    )

    if has_explanation(
        task_row,
        best_agent
    ):
        explanation_count += 1

In [ ]:
explanation_coverage = (
    explanation_count
    / len(test_data)
)

print(
    "Explanation coverage:",
    round(
        explanation_coverage * 100,
        2
    ),
    "%"
)

Explanation coverage: 100.0 %


## 18. Saving Evaluation Results

The final evaluation results and selected hybrid weights are saved so that they can be used in the implementation report and during website integration.

In [ ]:
comparison_df.to_csv(
    os.path.join(
        MODEL_PATH,
        "model_comparison.csv"
    ),
    index=False
)

In [ ]:
tuning_results_df.to_csv(
    os.path.join(
        MODEL_PATH,
        "hybrid_weight_tuning.csv"
    ),
    index=False
)

In [ ]:
weights_file = os.path.join(
    MODEL_PATH,
    "best_hybrid_weights.json"
)

with open(
    weights_file,
    "w"
) as file:

    json.dump(
        BEST_WEIGHTS,
        file,
        indent=4
    )

In [ ]:
evaluation_summary = {
    "Precision@5":
        float(
            final_test_metrics[
                "Precision@5"
            ]
        ),

    "Recall@5":
        float(
            final_test_metrics[
                "Recall@5"
            ]
        ),

    "MAP@5":
        float(
            final_test_metrics[
                "MAP@5"
            ]
        ),

    "Explanation_Coverage":
        float(
            explanation_coverage
        ),

    "KG_Workload_Std":
        float(
            kg_workload_std
        ),

    "Hybrid_Workload_Std":
        float(
            hybrid_workload_std
        )
}

In [ ]:
with open(
    os.path.join(
        MODEL_PATH,
        "evaluation_summary.json"
    ),
    "w"
) as file:

    json.dump(
        evaluation_summary,
        file,
        indent=4
    )

print(
    "Evaluation results saved successfully"
)

Evaluation results saved successfully


# Notebook 03 Summary

The recommendation system was evaluated using validation and testing datasets.

Hybrid weights were tuned using validation data before the final model was evaluated on unseen testing records.

The final evaluation included:

- Precision@5
- Recall@5
- MAP@5
- Comparison with individual recommendation components
- Recommendation workload distribution
- Explanation coverage

The Knowledge Graph component achieved the strongest ranking performance on the testing dataset, while the Hybrid model combined recommendation signals with operational considerations such as availability, workload and performance.

The results are interpreted as prototype experimental findings because the historical agent-task assignments used in the experiment were synthetically generated.

In [ ]:
print("NOTEBOOK 03 COMPLETE")

print(
    "\nHybrid Precision@5:",
    round(
        final_test_metrics[
            "Precision@5"
        ],
        4
    )
)

print(
    "Hybrid Recall@5:",
    round(
        final_test_metrics[
            "Recall@5"
        ],
        4
    )
)

print(
    "Hybrid MAP@5:",
    round(
        final_test_metrics[
            "MAP@5"
        ],
        4
    )
)

print(
    "\nExplanation coverage:",
    round(
        explanation_coverage * 100,
        2
    ),
    "%"
)

print(
    "\nKnowledge Graph workload std:",
    round(
        kg_workload_std,
        2
    )
)

print(
    "Hybrid workload std:",
    round(
        hybrid_workload_std,
        2
    )
)

NOTEBOOK 03 COMPLETE

Hybrid Precision@5: 0.07
Hybrid Recall@5: 0.3499
Hybrid MAP@5: 0.2186

Explanation coverage: 100.0 %

Knowledge Graph workload std: 106.06
Hybrid workload std: 80.72


In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity


# -------------------------
# Load saved files
# -------------------------

BASE_PATH = (
    "/content/drive/MyDrive/"
    "task-allocation-recommender"
)

PROCESSED_PATH = os.path.join(
    BASE_PATH,
    "data",
    "onboarding",
    "processed"
)

SPLITS_PATH = os.path.join(
    BASE_PATH,
    "data",
    "onboarding",
    "splits"
)

MODEL_PATH = os.path.join(
    BASE_PATH,
    "models"
)


tasks = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "onboarding_tasks.parquet"
    )
)

test_data = pd.read_parquet(
    os.path.join(
        SPLITS_PATH,
        "test.parquet"
    )
)

components = joblib.load(
    os.path.join(
        MODEL_PATH,
        "recommender_components.joblib"
    )
)

with open(
    os.path.join(
        MODEL_PATH,
        "best_hybrid_weights.json"
    ),
    "r"
) as file:

    BEST_WEIGHTS = json.load(file)


# -------------------------
# Restore components
# -------------------------

tfidf_vectorizer = (
    components[
        "tfidf_vectorizer"
    ]
)

agent_skill_vectors = (
    components[
        "agent_skill_vectors"
    ]
)

agents = (
    components[
        "agents"
    ].copy()
)

collaborative_scores = (
    components[
        "collaborative_scores"
    ]
)

profile_to_column = (
    components[
        "profile_to_column"
    ]
)

historical_task_types = (
    components[
        "historical_task_types"
    ]
)


tasks["task_id"] = (
    tasks["task_id"].astype(str)
)

test_data["task_id"] = (
    test_data["task_id"].astype(str)
)

test_data["agent_id"] = (
    test_data["agent_id"].astype(str)
)

agents["agent_id"] = (
    agents["agent_id"].astype(str)
)


# -------------------------
# Operational scores
# -------------------------

agents["availability_score"] = (
    agents["availability_status"]
    .astype(str)
    .str.lower()
    .eq("available")
    .astype(float)
)

max_workload = (
    agents["current_workload"].max()
)

if max_workload == 0:
    max_workload = 1

agents["workload_score"] = (
    1
    - (
        agents["current_workload"]
        / max_workload
    )
)

agents["performance_score"] = (
    agents["performance_score"]
    .fillna(
        agents[
            "performance_score"
        ].mean()
    )
    .clip(0, 1)
)

agents["operational_score"] = (
    0.40
    * agents["availability_score"]

    + 0.35
    * agents["workload_score"]

    + 0.25
    * agents["performance_score"]
)


# -------------------------
# Helper functions
# -------------------------

def split_skills(text):

    if pd.isna(text):
        return []

    return [
        x.strip()
        for x in str(text).split(",")
        if x.strip()
    ]


def skill_match(
    task_skills,
    agent_skills
):

    required = set(
        split_skills(task_skills)
    )

    available = set(
        split_skills(agent_skills)
    )

    if len(required) == 0:
        return 0

    return (
        len(
            required.intersection(
                available
            )
        )
        / len(required)
    )


task_lookup = (
    tasks.set_index(
        "task_id",
        drop=False
    )
)


def score_task(task):

    # Content
    task_vector = (
        tfidf_vectorizer.transform(
            [
                str(
                    task[
                        "required_skills"
                    ]
                )
            ]
        )
    )

    content = (
        cosine_similarity(
            task_vector,
            agent_skill_vectors
        )
        .flatten()
    )

    # Collaborative
    profile = task[
        "task_profile"
    ]

    if profile in profile_to_column:

        cf = collaborative_scores[
            :,
            profile_to_column[
                profile
            ]
        ]

    else:

        cf = np.zeros(
            len(agents)
        )

    # Knowledge Graph
    skill_scores = np.array(
        [
            skill_match(
                task[
                    "required_skills"
                ],
                x
            )
            for x in agents["skills"]
        ]
    )

    location_scores = (
        agents["county"]
        .astype(str)
        .eq(
            str(
                task["county"]
            )
        )
        .astype(float)
        .to_numpy()
    )

    history = (
        historical_task_types[
            historical_task_types[
                "task_type"
            ]
            ==
            task[
                "task_type"
            ]
        ]
        .set_index(
            "agent_id"
        )["count"]
    )

    history_counts = (
        agents["agent_id"]
        .map(history)
        .fillna(0)
        .to_numpy()
    )

    max_history = (
        history_counts.max()
    )

    if max_history == 0:
        max_history = 1

    type_score = (
        history_counts
        / max_history
    )

    kg = (
        0.60 * skill_scores
        + 0.20 * location_scores
        + 0.20 * type_score
    )

    operational = (
        agents[
            "operational_score"
        ].to_numpy()
    )

    return (
        BEST_WEIGHTS["content"]
        * content

        + BEST_WEIGHTS[
            "collaborative"
        ]
        * cf

        + BEST_WEIGHTS[
            "knowledge_graph"
        ]
        * kg

        + BEST_WEIGHTS[
            "operational"
        ]
        * operational
    )


# -------------------------
# Calculate Top-1 accuracy
# -------------------------

correct_top1 = 0
score_cache = {}

for _, record in test_data.iterrows():

    task = task_lookup.loc[
        str(record["task_id"])
    ]

    cache_key = (
        str(task["task_profile"]),
        str(task["task_type"]),
        str(task["county"]),
        str(task["required_skills"])
    )

    if cache_key not in score_cache:

        score_cache[
            cache_key
        ] = score_task(task)

    scores = score_cache[
        cache_key
    ]

    best_index = int(
        np.argmax(scores)
    )

    predicted_agent = str(
        agents.iloc[
            best_index
        ]["agent_id"]
    )

    actual_agent = str(
        record["agent_id"]
    )

    if predicted_agent == actual_agent:
        correct_top1 += 1


top1_accuracy = (
    correct_top1
    / len(test_data)
    * 100
)

print(
    "Correct Top-1 predictions:",
    correct_top1
)

print(
    "Total test tasks:",
    len(test_data)
)

print(
    "Top-1 Accuracy:",
    round(top1_accuracy, 2),
    "%"
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/task-allocation-recommender/data/onboarding/processed/onboarding_tasks.parquet'